# Classification avec des Support Vector Machines (SVM)

**Module :** Machine Learning 1 — ECE Paris, 2025/2026  
**Auteur :** Édouard Menut

Objectif : construire et comparer des classifieurs SVM sur quatre jeux de données UCI de difficulté croissante, en travaillant la chaîne complète : nettoyage, encodage, normalisation, sélection de variables, validation croisée et recherche d'hyperparamètres.

| Jeu de données | Observations | Variables | Type | Cible |
|---|---|---|---|---|
| Breast Cancer Wisconsin | 699 | 9 | numériques (1–10) | bénin / malin |
| Spambase | 4 601 | 57 | numériques (fréquences) | spam / non-spam |
| Car Evaluation | 1 728 | 6 | catégorielles | 4 niveaux d'acceptabilité |
| Mushroom | 8 124 | 22 | catégorielles | comestible / vénéneux |

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report

## Fonctions utilitaires

Trois fonctions pour rendre l'analyse répétable sur chaque jeu de données :
- `do_cv` : validation croisée stratifiée à 5 plis mélangés (moyenne et écart-type de l'accuracy)
- `do_cm_cr` : matrice de confusion et rapport de classification sur un split stratifié 67/33
- `do_grid_search` : recherche du meilleur noyau et du meilleur `C`

In [2]:
# Les fichiers UCI sont triés (par classe ou par modalité) : on mélange les plis
# de validation croisée, sinon certains plis ne contiennent presque qu'une seule classe.
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def do_cv(model, X, y, cv=CV):
    scores = cross_val_score(model, X, y, cv=cv)
    print(f"{model}\n  scores CV : {np.round(scores, 4)}")
    print(f"  accuracy moyenne : {scores.mean():.4f} (± {scores.std():.4f})")
    return scores.mean(), scores.std()

def do_cm_cr(model, X, y, names):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.33, stratify=y, random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Matrice de confusion :")
    print(pd.DataFrame(confusion_matrix(y_test, y_pred), index=names, columns=names))
    print("\nRapport de classification :")
    print(classification_report(y_test, y_pred, target_names=names, zero_division=0))

def do_grid_search(model, parameters, X, y, cv=CV, scoring="accuracy"):
    grid = GridSearchCV(model, parameters, cv=cv, scoring=scoring, n_jobs=-1)
    grid.fit(X, y)
    print("Meilleurs paramètres :", grid.best_params_)
    print(f"Meilleur score ({scoring}) : {grid.best_score_:.4f}")
    return grid.best_estimator_

def select_k_best_features(X, y, k):
    # chi2 impose des valeurs positives : on ramène d'abord chaque variable dans [0, 1]
    X_scaled = MinMaxScaler().fit_transform(X)
    selector = SelectKBest(score_func=chi2, k=k)
    return selector.fit_transform(X_scaled, y), selector

---
## 1. Breast Cancer Wisconsin

### 1.a Chargement et nettoyage

In [3]:
df_bc = pd.read_csv("data/breast_cancer.csv")
df_bc.dtypes

Sample_code_number             int64
Clump_Thickness                int64
Uniformity_of_Cell_Size        int64
Uniformity_of_Cell_Shape       int64
Marginal_Adhesion              int64
Single_Epithelial_Cell_Size    int64
Bare_Nuclei                      str
Bland_Chromatin                int64
Normal_Nucleoli                int64
Mitoses                        int64
Class                          int64
dtype: object

`Bare_Nuclei` est lue comme du texte car les valeurs manquantes sont codées `?`. On la convertit en numérique, on supprime les 16 lignes incomplètes, on retire l'identifiant `Sample_code_number` (aucune valeur prédictive) et on encode la cible : 2 (bénin) → 0, 4 (malin) → 1.

In [4]:
df_bc["Bare_Nuclei"] = pd.to_numeric(df_bc["Bare_Nuclei"], errors="coerce")
print("Valeurs manquantes :", df_bc["Bare_Nuclei"].isna().sum())

df_bc = df_bc.dropna().drop(columns="Sample_code_number")
df_bc["Class"] = df_bc["Class"].map({2: 0, 4: 1})

X = df_bc.drop(columns="Class")
y = df_bc["Class"]
print("X :", X.shape)
print("Répartition de la cible :\n", y.value_counts(normalize=True).round(3))

Valeurs manquantes : 16
X : (683, 9)
Répartition de la cible :
 Class
0    0.65
1    0.35
Name: proportion, dtype: float64


La baseline (toujours prédire « bénin ») donne environ **65 %** d'accuracy : c'est le score à battre.

### 1.b Modèles : SVM linéaire vs noyau RBF, avec et sans normalisation

In [5]:
print("Sans normalisation")
do_cv(LinearSVC(max_iter=10000, random_state=42), X, y)
do_cv(SVC(kernel="rbf"), X, y)

print("\nAvec StandardScaler")
X_scaled = StandardScaler().fit_transform(X)
do_cv(LinearSVC(max_iter=10000, random_state=42), X_scaled, y)
do_cv(SVC(kernel="rbf"), X_scaled, y);

Sans normalisation
LinearSVC(max_iter=10000, random_state=42)
  scores CV : [0.9781 0.9562 0.9708 0.9559 0.9706]
  accuracy moyenne : 0.9663 (± 0.0088)
SVC()
  scores CV : [0.9781 0.9708 0.9781 0.9706 0.9632]
  accuracy moyenne : 0.9722 (± 0.0056)

Avec StandardScaler
LinearSVC(max_iter=10000, random_state=42)
  scores CV : [0.9781 0.9562 0.9781 0.9559 0.9706]
  accuracy moyenne : 0.9678 (± 0.0100)
SVC()
  scores CV : [0.9635 0.9708 0.9708 0.9706 0.9632]
  accuracy moyenne : 0.9678 (± 0.0036)


In [6]:
do_cm_cr(SVC(kernel="rbf"), X_scaled, y, names=["bénin", "malin"])

Matrice de confusion :
       bénin  malin
bénin    142      5
malin      3     76

Rapport de classification :


              precision    recall  f1-score   support

       bénin       0.98      0.97      0.97       147
       malin       0.94      0.96      0.95        79

    accuracy                           0.96       226
   macro avg       0.96      0.96      0.96       226
weighted avg       0.96      0.96      0.96       226



Dans un contexte de diagnostic médical, les **faux négatifs** (tumeur maligne prédite bénigne) sont l'erreur la plus grave : on préfère un modèle qui génère quelques faux positifs (examens complémentaires) plutôt que de manquer une tumeur. Le rappel de la classe « malin » est donc la métrique à surveiller.

### 1.c Sélection de variables (chi²)

In [7]:
X_top5, selector = select_k_best_features(X, y, k=5)
print("5 meilleures variables :", list(X.columns[selector.get_support()]))

do_cv(LinearSVC(max_iter=10000, random_state=42), X_top5, y)
do_cv(SVC(kernel="rbf"), X_top5, y);

5 meilleures variables : ['Uniformity_of_Cell_Size', 'Uniformity_of_Cell_Shape', 'Marginal_Adhesion', 'Bare_Nuclei', 'Normal_Nucleoli']
LinearSVC(max_iter=10000, random_state=42)
  scores CV : [0.9708 0.9489 0.9562 0.9559 0.9706]
  accuracy moyenne : 0.9605 (± 0.0087)
SVC()
  scores CV : [0.9781 0.9489 0.9489 0.9706 0.9706]
  accuracy moyenne : 0.9634 (± 0.0122)


### 1.d Recherche d'hyperparamètres

In [8]:
param_grid = {"kernel": ["linear", "rbf", "poly"], "C": [0.01, 0.1, 1, 10, 100]}
best_bc = do_grid_search(SVC(), param_grid, X_scaled, y)

Meilleurs paramètres : {'C': 0.01, 'kernel': 'linear'}
Meilleur score (accuracy) : 0.9707


Sur ce jeu de données, les modèles linéaire et RBF sont au coude-à-coude (~97 %) : les écarts sont inférieurs à l'écart-type de la validation croisée, donc non significatifs. Les données sont presque linéairement séparables, un SVM linéaire fortement régularisé (`C=0.01`) suffit. Réduire à 5 variables fait perdre moins d'un point : le modèle peut être simplifié sans vraie perte.

---
## 2. Spambase — beaucoup de variables

57 variables numériques (fréquences de mots et de caractères, longueur des séquences en majuscules). On se concentre sur la sélection des 15 variables les plus discriminantes.

In [9]:
df_spam = pd.read_csv("data/spambase.csv")
X = df_spam.drop(columns="class")
y = df_spam["class"]
print("X :", X.shape, "| part de spam :", round(y.mean(), 3))

X_top15, selector = select_k_best_features(X, y, k=15)
print("15 meilleures variables :", list(X.columns[selector.get_support()]))

X : (4601, 57) | part de spam : 0.394
15 meilleures variables : ['word_freq_our', 'word_freq_remove', 'word_freq_order', 'word_freq_receive', 'word_freq_addresses', 'word_freq_free', 'word_freq_business', 'word_freq_your', 'word_freq_000', 'word_freq_money', 'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_labs', 'char_freq_$']


In [10]:
param_grid = {"kernel": ["linear", "rbf"], "C": [0.01, 1, 100]}
best_spam = do_grid_search(SVC(), param_grid, X_top15, y)
do_cv(best_spam, X_top15, y)
do_cm_cr(best_spam, X_top15, y, names=["non-spam", "spam"])

Meilleurs paramètres : {'C': 100, 'kernel': 'rbf'}
Meilleur score (accuracy) : 0.9061


SVC(C=100)
  scores CV : [0.8969 0.9196 0.9054 0.9011 0.9076]
  accuracy moyenne : 0.9061 (± 0.0077)


Matrice de confusion :
          non-spam  spam
non-spam       871    49
spam           106   493

Rapport de classification :
              precision    recall  f1-score   support

    non-spam       0.89      0.95      0.92       920
        spam       0.91      0.82      0.86       599

    accuracy                           0.90      1519
   macro avg       0.90      0.88      0.89      1519
weighted avg       0.90      0.90      0.90      1519



Avec seulement 15 variables sur 57, le SVM RBF atteint ~91 % d'accuracy. Le rappel du spam (0,82) est plus faible que celui des mails légitimes (0,95) : le modèle laisse passer quelques spams mais bloque rarement un vrai mail, ce qui est le compromis souhaitable pour un filtre anti-spam.

---
## 3. Car Evaluation — variables catégorielles, cible à 4 classes

Toutes les variables sont catégorielles (`vhigh`, `med`, `small`…). Un SVM travaille sur des distances : on utilise un **one-hot encoding** pour ne pas introduire d'ordre artificiel entre les modalités. La cible (`unacc`, `acc`, `good`, `vgood`) est encodée avec un `LabelEncoder` ; `SVC` gère le multi-classe en one-vs-one.

In [11]:
df_car = pd.read_csv("data/car.csv")
print(df_car["acceptability"].value_counts())

X = df_car.drop(columns="acceptability")
y = df_car["acceptability"]

ohe = OneHotEncoder(sparse_output=False)
X_car = ohe.fit_transform(X)
le = LabelEncoder()
y_car = le.fit_transform(y)
print("X après one-hot :", X_car.shape, "| classes :", list(le.classes_))

acceptability
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64
X après one-hot : (1728, 21) | classes : ['acc', 'good', 'unacc', 'vgood']


La cible est très déséquilibrée (70 % de `unacc`) : l'accuracy seule est trompeuse, on regarde aussi le rapport par classe.

In [12]:
param_grid = {"kernel": ["linear", "rbf"], "C": [0.1, 1, 10, 100]}
best_car = do_grid_search(SVC(), param_grid, X_car, y_car)
do_cm_cr(best_car, X_car, y_car, names=list(le.classes_))

Meilleurs paramètres : {'C': 10, 'kernel': 'rbf'}
Meilleur score (accuracy) : 0.9994
Matrice de confusion :
       acc  good  unacc  vgood
acc    127     0      0      0
good     0    23      0      0
unacc    1     0    399      0
vgood    0     0      0     21

Rapport de classification :
              precision    recall  f1-score   support

         acc       0.99      1.00      1.00       127
        good       1.00      1.00      1.00        23
       unacc       1.00      1.00      1.00       400
       vgood       1.00      1.00      1.00        21

    accuracy                           1.00       571
   macro avg       1.00      1.00      1.00       571
weighted avg       1.00      1.00      1.00       571



Avec un one-hot encoding et des plis de validation mélangés, le SVM RBF classe quasiment parfaitement les 4 niveaux (99,9 %), y compris les classes rares `good` et `vgood`.

**Point de vigilance :** avec des plis non mélangés (`cv=3` par défaut), le score tombe à ~70 % car le fichier est trié et certains plis ne contiennent aucun exemple des classes rares. C'est un piège classique de la validation croisée sur des données ordonnées.

In [13]:
# Même démarche avec uniquement les 10 puis 5 meilleures variables one-hot
for k in (10, 5):
    X_k = SelectKBest(chi2, k=k).fit_transform(X_car, y_car)
    print(f"--- {k} meilleures variables")
    do_grid_search(SVC(), param_grid, X_k, y_car)

--- 10 meilleures variables


Meilleurs paramètres : {'C': 1, 'kernel': 'linear'}
Meilleur score (accuracy) : 0.8472
--- 5 meilleures variables


Meilleurs paramètres : {'C': 1, 'kernel': 'linear'}
Meilleur score (accuracy) : 0.7922


Contrairement aux jeux précédents, retirer des variables dégrade nettement le modèle (99,9 % → 85 % → 79 %) : chaque caractéristique de la voiture apporte une information utile à la décision.

---
## 4. Mushroom — 8 124 champignons, 22 variables catégorielles

Même stratégie : one-hot encoding des variables, cible binaire (e = comestible, p = vénéneux). On vérifie ensuite si quelques variables suffisent.

In [14]:
df_mush = pd.read_csv("data/mushroom.csv")
y = LabelEncoder().fit_transform(df_mush["class"])   # e -> 0, p -> 1
X = df_mush.drop(columns="class")

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_mush = ohe.fit_transform(X)
print("X après one-hot :", X_mush.shape)

X après one-hot : (8124, 117)


In [15]:
param_grid = {"kernel": ["linear", "rbf"], "C": [0.01, 0.1, 1]}
best_mush = do_grid_search(SVC(), param_grid, X_mush, y)
do_cv(best_mush, X_mush, y)
do_cm_cr(best_mush, X_mush, y, names=["comestible", "vénéneux"])

Meilleurs paramètres : {'C': 1, 'kernel': 'linear'}
Meilleur score (accuracy) : 1.0000


SVC(C=1, kernel='linear')
  scores CV : [1. 1. 1. 1. 1.]
  accuracy moyenne : 1.0000 (± 0.0000)
Matrice de confusion :
            comestible  vénéneux
comestible        1389         0
vénéneux             0      1292

Rapport de classification :
              precision    recall  f1-score   support

  comestible       1.00      1.00      1.00      1389
    vénéneux       1.00      1.00      1.00      1292

    accuracy                           1.00      2681
   macro avg       1.00      1.00      1.00      2681
weighted avg       1.00      1.00      1.00      2681



In [16]:
# Avec seulement 5 variables one-hot
selector = SelectKBest(chi2, k=5).fit(X_mush, y)
print("5 meilleures variables :", list(ohe.get_feature_names_out()[selector.get_support()]))
X_top5 = selector.transform(X_mush)
best_mush5 = do_grid_search(SVC(), param_grid, X_top5, y)
do_cm_cr(best_mush5, X_top5, y, names=["comestible", "vénéneux"])

5 meilleures variables : ['odor_f', 'odor_n', 'gill-color_b', 'stalk-surface-above-ring_k', 'stalk-surface-below-ring_k']


Meilleurs paramètres : {'C': 0.01, 'kernel': 'rbf'}
Meilleur score (accuracy) : 0.9301


Matrice de confusion :
            comestible  vénéneux
comestible        1389         0
vénéneux           185      1107

Rapport de classification :
              precision    recall  f1-score   support

  comestible       0.88      1.00      0.94      1389
    vénéneux       1.00      0.86      0.92      1292

    accuracy                           0.93      2681
   macro avg       0.94      0.93      0.93      2681
weighted avg       0.94      0.93      0.93      2681



Avec seulement 5 indicateurs (dont l'odeur), le modèle reste à ~93 %, mais **185 champignons vénéneux sont classés comestibles** : aucune précision n'est acceptable si l'erreur coûteuse n'est pas maîtrisée. Le modèle complet (117 variables one-hot, noyau linéaire) atteint 100 %.

---
## Conclusion

| Jeu de données | Meilleur modèle | Accuracy (CV 5 plis) |
|---|---|---|
| Breast Cancer | SVM linéaire, C = 0.01 (données standardisées) | 97,1 % |
| Spambase (15 variables) | SVM RBF, C = 100 | 90,6 % |
| Car Evaluation (one-hot) | SVM RBF, C = 10 | 99,9 % |
| Mushroom (one-hot) | SVM linéaire, C = 1 | 100 % |

**Enseignements :**
- La **préparation des données** compte autant que le modèle : valeurs manquantes codées `?`, identifiant à retirer, encodage one-hot des variables catégorielles.
- La **validation croisée doit être stratifiée et mélangée** sur des fichiers triés, sinon les scores sont faussés.
- La **sélection de variables** simplifie le modèle sans perte quand l'information est redondante (Breast Cancer), mais peut être très coûteuse sinon (Car, Mushroom).
- Le choix de la métrique dépend du **coût des erreurs** : faux négatifs en diagnostic, champignons vénéneux prédits comestibles.